# Semi-annual Model Inventory Attestation

**Step 1** – Clean master copy: keeps only the nine attestation tabs, converts formulas to values, removes external links, adds a `Metadata` tab.

**Step 1b** – In-scope copy: applies the scope filters (e.g. no retired models, open limitations only). Owner files are built from this copy; the full master is kept as evidence.

**Step 2** – One Excel file per owner (PLC Model Owner / PLC DQM Owner / PMA Owner), filtered via `Model ID`, `DQM ID` and `PMA ID`. A person who owns models, DQMs and PMAs gets one file with all relevant tabs; tab groups they own nothing in are dropped.

Only edit the **Configuration** cell, then run all cells top to bottom (Kernel ▸ Restart & Run All).

Requires: `openpyxl` (2.6 or later; newest version your Python allows is recommended), `pandas`. Python 3.7+.

## 1. Configuration

In [ ]:
from pathlib import Path

# --- Paths -------------------------------------------------------------
INPUT_FILE = Path(r"C:\MRM\Inventory\MRM_Model_Inventory.xlsx")   # current inventory
OUTPUT_DIR = Path(r"C:\MRM\Attestation\2026_H2")                  # created if missing

# --- Tabs to include in the attestation (in this order) ---------------
ATTESTATION_SHEETS = [
    "Model Inventory", "Model Limitation", "Model Validations",
    "DQM Inventory",   "DQM Limitation",   "DQM Reviews",
    "PMA Proposals",   "PMA Limitation",   "PMA Validations",
]

# --- How owners are linked to each tab group --------------------------
GROUPS = {
    "Model": {"inventory_sheet": "Model Inventory", "owner_col": "PLC Model Owner", "id_col": "Model ID",
              "sheets": ["Model Inventory", "Model Limitation", "Model Validations"]},
    "DQM":   {"inventory_sheet": "DQM Inventory",   "owner_col": "PLC DQM Owner",   "id_col": "DQM ID",
              "sheets": ["DQM Inventory", "DQM Limitation", "DQM Reviews"]},
    "PMA":   {"inventory_sheet": "PMA Proposals",   "owner_col": "PMA Owner",       "id_col": "PMA ID",
              "sheets": ["PMA Proposals", "PMA Limitation", "PMA Validations"]},
}

# --- Scope filters (applied after the MASTER copy, before the owner files) ---
# (tab, column, "equals" or "not equals", value or [list of values])
# Rows failing a filter are removed. Several filters on the same tab are all applied.
SCOPE_FILTERS = [
    ("Model Inventory",  "Current Model Status", "not equals", "9. Retired"),
    ("Model Limitation", "Status",               "equals",     "Open"),
    ("DQM Inventory",    "Current DQM Status",   "not equals", "9. Retired"),
    ("DQM Limitation",   "Status",               "equals",     "Open"),
    ("PMA Proposals",    "Is Live",              "equals",     "Yes"),
    ("PMA Limitation",   "Finding Status",       "equals",     "Open"),
]

# How each group's scope is described on the cover page
SCOPE_WORDING = {"Model": "live or in-development model", "DQM": "non-retired DQM", "PMA": "live PMA"}

# If a tab names its ID column differently, map it here, e.g. {"Model Validations": "Model Ref"}
ID_COLUMN_OVERRIDES = {}

# If one owner cell can hold several names, give the separator(s) as a regex, e.g. r"[;/\n]"
# Leave as None to treat each cell as a single name.
OWNER_SEPARATOR = None

# --- Cover page (Attestation tab) -------------------------------------
ATTESTATION_PERIOD = "H2 2026"
RETURN_BY          = "20 November 2026"
MRM_CONTACT        = "UKRM Model Risk Management"     # who owners return the file to
RESPONSE_OPTIONS   = ["Agree", "Agree - changes made in tab", "Do not agree"]
PROTECT_COVER      = True    # lock the cover page except the input cells (no password)

REMOVE_HYPERLINKS  = True   # strip cell hyperlinks (e.g. links to SharePoint docs)
HEADER_SEARCH_ROWS = 20     # header row is detected automatically within the first N rows

## 2. Helper functions (no need to edit)

In [ ]:
import re
import getpass
from datetime import datetime

import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter, range_boundaries
import openpyxl
try:                                   # openpyxl 3.1+
    from openpyxl.worksheet.formula import ArrayFormula, DataTableFormula
    FORMULA_OBJECTS = (ArrayFormula, DataTableFormula)
except ImportError:                    # openpyxl 3.0.x: array formulas are plain strings starting with "="
    FORMULA_OBJECTS = ()
print(f"openpyxl version: {openpyxl.__version__}")
if openpyxl.__version__ == "3.1.3":
    print("WARNING: openpyxl 3.1.3 has a bug that can stop it re-opening files it saved. "
          "Run  %pip install openpyxl==3.1.2  and restart the kernel.")

RUN_TS = datetime.now()
STAMP = RUN_TS.strftime("%Y%m%d")
KEPT = set(ATTESTATION_SHEETS)


def norm_text(v):
    """Case- and whitespace-insensitive key, so 'John  Doe' == 'john doe'."""
    return " ".join(str(v).split()).casefold() if v is not None else ""


def norm_id(v):
    """Makes 101, 101.0 and '101' match each other."""
    if v is None:
        return ""
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return str(v).strip().casefold()


def split_owners(v):
    if v is None or str(v).strip() == "":
        return []
    parts = re.split(OWNER_SEPARATOR, str(v)) if OWNER_SEPARATOR else [str(v)]
    return [" ".join(p.split()) for p in parts if p.strip()]


def id_col_for(sheet, group_cfg):
    return ID_COLUMN_OVERRIDES.get(sheet, group_cfg["id_col"])


def find_header(ws, col_name):
    """Returns (header_row, column_index) of col_name within the first HEADER_SEARCH_ROWS rows."""
    target = norm_text(col_name)
    for row in ws.iter_rows(min_row=1, max_row=min(HEADER_SEARCH_ROWS, ws.max_row)):
        for c in row:
            if norm_text(c.value) == target:
                return c.row, c.column
    raise ValueError(f"Column '{col_name}' not found in the first {HEADER_SEARCH_ROWS} rows of tab '{ws.title}'.")


def formula_refs_ok(formula, valid_names, kept):
    """False if a formula points at a removed tab, an external file or a removed named range."""
    if not formula:
        return True
    f = str(formula).lstrip("=").strip()
    if f.startswith('"'):
        return True                                   # literal list, e.g. "Yes,No"
    if "[" in f:
        return False                                  # external workbook
    refs = re.findall(r"(?:'((?:[^']|'')+)'|([A-Za-z0-9_\.]+))!", f)
    for quoted, plain in refs:
        if (quoted or plain).replace("''", "'") not in kept:
            return False
    if not refs and re.fullmatch(r"[A-Za-z_\\][\w\.]*", f) and not re.fullmatch(r"\$?[A-Za-z]{1,3}\$?\d+", f):
        return f.casefold() in valid_names            # bare named range
    return True


def clean_references(wb):
    """Drops named ranges and dropdown lists that point at tabs no longer in the workbook or at other files."""
    kept = set(wb.sheetnames)
    if hasattr(wb.defined_names, "items"):              # openpyxl 3.1+
        for name, dn in list(wb.defined_names.items()):
            if not formula_refs_ok(dn.attr_text, set(), kept):
                del wb.defined_names[name]
        valid_names = {n.casefold() for n in wb.defined_names}
        for ws in wb.worksheets:
            for name, dn in list(ws.defined_names.items()):
                if not formula_refs_ok(dn.attr_text, valid_names, kept):
                    del ws.defined_names[name]
    else:                                               # openpyxl 3.0.x
        wb.defined_names.definedName = [
            dn for dn in wb.defined_names.definedName
            if dn.localSheetId is None and formula_refs_ok(dn.attr_text, set(), kept)
        ]
        valid_names = {dn.name.casefold() for dn in wb.defined_names.definedName}
    for ws in wb.worksheets:
        ws.data_validations.dataValidation = [
            dv for dv in ws.data_validations.dataValidation
            if formula_refs_ok(dv.formula1, valid_names, kept) and formula_refs_ok(dv.formula2, valid_names, kept)
        ]


def safe_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "", name).strip().replace(" ", "_")[:80] or "Unknown"


def reset_tab_selection(wb):
    for ws in wb.worksheets:
        ws.sheet_view.tabSelected = False
    wb.active = 0
    wb.worksheets[0].sheet_view.tabSelected = True


# Statement shown against each tab on the cover page
TAB_STATEMENTS = {
    "Inventory":   "The details recorded for each {item} listed are accurate and up to date.",
    "Proposals":   "The details recorded for each {item} listed are accurate and up to date.",
    "Limitation":  "All open limitations/findings are listed, and their descriptions and status are accurate.",
    "Validations": "The validation history and outcomes shown are accurate.",
    "Reviews":     "The review history and outcomes shown are accurate.",
}
ITEM_WORD = {"Model": "model", "DQM": "DQM", "PMA": "PMA"}


def add_cover_page(wb, owner_name, owned, row_counts):
    """Inserts an 'Attestation' tab at the front of an owner file.
    owned: {group: set(ids)}; row_counts: {tab: number of rows kept}."""
    from openpyxl.styles import Alignment, Border, Side, Protection
    from openpyxl.worksheet.datavalidation import DataValidation
    from openpyxl.formatting.rule import CellIsRule, FormulaRule

    ws = wb.create_sheet("Attestation", 0)
    navy, grey = "1F4E78", "D9D9D9"
    input_fill = PatternFill("solid", fgColor="FFF2CC")             # light yellow = owner fills in
    hdr_font, hdr_fill = Font(bold=True, color="FFFFFF"), PatternFill("solid", fgColor=navy)
    thin = Side(style="thin", color="A6A6A6")
    box = Border(left=thin, right=thin, top=thin, bottom=thin)
    wrap = Alignment(wrap_text=True, vertical="top")
    unlocked = Protection(locked=False)

    for col, width in zip("ABCDE", [30, 10, 58, 28, 60]):
        ws.column_dimensions[col].width = width

    def merged(row, text, font=None, height=None, cols="A{0}:E{0}"):
        ws.merge_cells(cols.format(row))
        c = ws.cell(row, 1, text)
        c.alignment = wrap
        if font:
            c.font = font
        if height:
            ws.row_dimensions[row].height = height
        return c

    def input_cell(cell):
        cell.fill, cell.border, cell.protection = input_fill, box, unlocked
        cell.alignment = wrap

    # --- Title & instructions
    merged(1, f"Model Inventory Attestation - {ATTESTATION_PERIOD}", Font(bold=True, size=16, color=navy), 26)
    merged(2, f"Owner: {owner_name}     |     Please return to {MRM_CONTACT} by {RETURN_BY}",
           Font(bold=True, size=11))
    merged(4, "How to complete this file", Font(bold=True, color=navy))
    steps = [
        "1. Review every tab. Where a value is wrong or out of date, overwrite it directly in the cell. "
        "Please do not change the ID columns (Model ID, DQM ID, PMA ID): MRM uses them to detect your changes.",
        "2. If an item you own is missing, add it as a new row at the bottom of the relevant tab and leave the ID blank.",
        "3. Do not delete rows. If you no longer own an item, or it has been retired, update its status and say so in the comments below. "
        "If an open limitation has been closed, change its status and explain in the comments.",
        "4. Answer each statement below (yellow cells), add comments where relevant, complete your details and return the file.",
    ]
    for i, text in enumerate(steps, start=5):
        merged(i, text, height=30)

    # --- Attester details
    r = 10
    merged(r, "Attester details", Font(bold=True, color=navy))
    for label in ["Attested by (name)", "Role", "Date of attestation"]:
        r += 1
        ws.cell(r, 1, label).font = Font(bold=True)
        ws.merge_cells(start_row=r, start_column=2, end_row=r, end_column=3)
        input_cell(ws.cell(r, 2))
    date_cell = ws.cell(r, 2)
    date_cell.number_format = "DD/MM/YYYY"
    dv_date = DataValidation(type="date", operator="greaterThan", formula1="DATE(2020,1,1)", allow_blank=True,
                             showErrorMessage=True, error="Please enter a date, e.g. 15/11/2026")
    ws.add_data_validation(dv_date)
    dv_date.add(date_cell.coordinate)

    # --- Statements table
    r += 2
    first_resp_row = r + 1
    for j, h in enumerate(["Tab", "Rows", "Statement", "Response", "Comments"], 1):
        c = ws.cell(r, j, h)
        c.font, c.fill, c.border = hdr_font, hdr_fill, box

    statements = []
    for group, cfg in GROUPS.items():
        if not owned.get(group):
            continue
        item = ITEM_WORD.get(group, group)
        for tab in cfg["sheets"]:
            key = next((k for k in TAB_STATEMENTS if tab.endswith(k)), None)
            text = TAB_STATEMENTS[key].format(item=item) if key else "The information in this tab is accurate."
            statements.append((tab, row_counts.get(tab, ""), text))
        scope = SCOPE_WORDING.get(group, item)
        statements.append((f"{group} completeness", len(owned[group]),
                           f"Every {scope} I own is listed, and no {item} is listed that I no longer own."))

    dv_resp = DataValidation(type="list", formula1='"' + ",".join(RESPONSE_OPTIONS) + '"', allow_blank=True,
                             showErrorMessage=True, error="Please pick a response from the list.")
    ws.add_data_validation(dv_resp)
    for tab, n, text in statements:
        r += 1
        for j, v in enumerate([tab, n, text], 1):
            c = ws.cell(r, j, v)
            c.border, c.alignment = box, wrap
            if j == 1 and tab in wb.sheetnames:                     # clickable link to the tab
                c.hyperlink = "#'{}'!A1".format(tab)
                c.font = Font(color="0563C1", underline="single")
        ws.cell(r, 2).alignment = Alignment(horizontal="center", vertical="top")
        input_cell(ws.cell(r, 4))
        input_cell(ws.cell(r, 5))
        dv_resp.add(ws.cell(r, 4).coordinate)
        ws.row_dimensions[r].height = 32
    last_resp_row = r
    resp_range = f"D{first_resp_row}:D{last_resp_row}"

    # colour the responses
    for value, colour in zip(RESPONSE_OPTIONS, ["C6EFCE", "FFEB9C", "FFC7CE"]):
        ws.conditional_formatting.add(resp_range, CellIsRule(operator="equal", formula=['"{}"'.format(value)],
                                                             fill=PatternFill("solid", bgColor=colour)))

    # --- Completion status (live formula so the owner sees what is missing)
    r += 2
    ws.cell(r, 1, "Completion status").font = Font(bold=True)
    ws.merge_cells(start_row=r, start_column=2, end_row=r, end_column=3)
    det = f"B{first_resp_row - 5}:B{first_resp_row - 3}"            # the three attester cells
    status = ws.cell(r, 2)
    status.value = ('=IF(COUNTBLANK({0})+COUNTBLANK({1})=0,"Complete - ready to return",'
                    '"Incomplete: "&COUNTBLANK({0})&" response(s) and "&COUNTBLANK({1})&" detail(s) missing")'
                    ).format(resp_range, det)
    status.font, status.border = Font(bold=True), box
    ws.conditional_formatting.add(status.coordinate, FormulaRule(
        formula=['LEFT({},8)="Complete"'.format(status.coordinate)], fill=PatternFill("solid", bgColor="C6EFCE")))
    ws.conditional_formatting.add(status.coordinate, FormulaRule(
        formula=['LEFT({},10)="Incomplete"'.format(status.coordinate)], fill=PatternFill("solid", bgColor="FFC7CE")))

    r += 2
    merged(r, "By completing this page I confirm that I have reviewed the information in this file and that, "
              "together with the changes and comments I have made, it is complete and accurate to the best of my knowledge.",
           Font(italic=True), 30)

    # --- Page set-up & protection
    ws.sheet_view.showGridLines = False
    ws.page_setup.orientation = "landscape"
    ws.page_setup.fitToWidth, ws.page_setup.fitToHeight = 1, 0
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    if PROTECT_COVER:
        ws.protection.sheet = True
        ws.protection.formatRows = False                          # owners can still resize rows for long comments
    return ws


def keep_rows_where(ws, col_name, test):
    """Keeps only the data rows below the header where test(value in col_name) is True (fully empty rows are
    dropped too). Formats, row heights, Excel tables and filters are kept intact. Returns rows kept."""
    hdr, id_col = find_header(ws, col_name)
    last = ws.max_row

    # merged cells below the header would be corrupted by row deletion
    for mr in list(ws.merged_cells.ranges):
        if mr.min_row > hdr:
            ws.unmerge_cells(str(mr))

    rows = range(hdr + 1, last + 1)
    def row_is_empty(r):
        return all(c.value in (None, "") for c in ws[r])
    keep = [r for r in rows if not row_is_empty(r) and test(ws.cell(r, id_col).value)]
    drop = [r for r in rows if r not in set(keep)]
    dims = [(ws.row_dimensions[r].height, ws.row_dimensions[r].hidden) for r in keep]

    # delete contiguous blocks, bottom-up
    blocks, start, prev = [], None, None
    for r in drop:
        if start is None:
            start = prev = r
        elif r == prev + 1:
            prev = r
        else:
            blocks.append((start, prev)); start = prev = r
    if start is not None:
        blocks.append((start, prev))
    for s, e in reversed(blocks):
        ws.delete_rows(s, e - s + 1)

    # re-apply row heights to the rows that moved up
    new_last = hdr + len(keep)
    for i, (h, hidden) in enumerate(dims):
        rd = ws.row_dimensions[hdr + 1 + i]
        rd.height, rd.hidden = h, hidden
    for r in range(new_last + 1, last + 1):
        if r in ws.row_dimensions:
            del ws.row_dimensions[r]

    # shrink Excel tables and autofilters to the new size
    tables = ws.tables.values() if hasattr(ws, "tables") else ws._tables   # openpyxl 2.6 keeps a list
    for t in tables:
        c1, r1, c2, r2 = range_boundaries(t.ref)
        if r1 <= hdr <= r2:
            t.ref = f"{get_column_letter(c1)}{r1}:{get_column_letter(c2)}{max(new_last, r1 + 1)}"
            if t.autoFilter is not None:
                t.autoFilter.ref = t.ref
    if ws.auto_filter.ref:
        c1, r1, c2, r2 = range_boundaries(ws.auto_filter.ref)
        ws.auto_filter.ref = f"{get_column_letter(c1)}{r1}:{get_column_letter(c2)}{max(new_last, r1 + 1)}"
    return len(keep)



def filter_sheet(ws, id_col_name, keep_ids):
    """Keeps only the rows whose ID is in keep_ids."""
    return keep_rows_where(ws, id_col_name, lambda v: norm_id(v) in keep_ids)


def write_metadata(ws, rows):
    ws.append(["Field", "Value"])
    for c in ws[1]:
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor="1F4E78")
    for r in rows:
        ws.append(list(r))
    ws.column_dimensions["A"].width = 32
    ws.column_dimensions["B"].width = 90

## 3. Step 1 – Create the clean master copy

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

wb      = load_workbook(INPUT_FILE, keep_links=False)                  # formats + formulas
wb_vals = load_workbook(INPUT_FILE, keep_links=False, data_only=True)  # last values Excel calculated

missing = [s for s in ATTESTATION_SHEETS if s not in wb.sheetnames]
if missing:
    raise ValueError(f"Tabs not found in {INPUT_FILE.name}: {missing}")

# 1a. Drop every other tab and put the nine in the configured order
for name in list(wb.sheetnames):
    if name not in KEPT:
        del wb[name]
wb._sheets.sort(key=lambda ws: ATTESTATION_SHEETS.index(ws.title))

# 1b. Formulas -> values (and strip hyperlinks)
n_formulas, no_cached = 0, []
for ws in wb.worksheets:
    vals = wb_vals[ws.title]
    for row in ws.iter_rows():
        for c in row:
            if c.data_type == "f" or (FORMULA_OBJECTS and isinstance(c.value, FORMULA_OBJECTS)):
                c.value = vals[c.coordinate].value
                n_formulas += 1
                if c.value is None:
                    no_cached.append(f"{ws.title}!{c.coordinate}")
            if REMOVE_HYPERLINKS and getattr(c, "hyperlink", None) is not None:
                c.hyperlink = None

# 1c. Remove named ranges / dropdowns that would now point at removed tabs or other files
clean_references(wb)

# 1d. Metadata tab
write_metadata(wb.create_sheet("Metadata"), [
    ("Date of copy",                 RUN_TS.strftime("%d/%m/%Y %H:%M")),
    ("File name copied",             INPUT_FILE.name),
    ("Full source path",             str(INPUT_FILE.resolve())),
    ("Tabs included",                ", ".join(ATTESTATION_SHEETS)),
    ("Formulas converted to values", n_formulas),
    ("Prepared by",                  getpass.getuser()),
])

reset_tab_selection(wb)
MASTER_FILE = OUTPUT_DIR / f"Inventory_Attestation_MASTER_{STAMP}.xlsx"
wb.save(MASTER_FILE)

print(f"Master copy saved: {MASTER_FILE}")
print(f"Formulas converted to values: {n_formulas}")
if no_cached:
    print(f"WARNING: {len(no_cached)} formula cells had no stored value (blank result, or file not "
          f"recalculated/saved in Excel). First few: {no_cached[:10]}")

## 4. Step 1b – Apply the scope filters
Check the table this cell prints: if a filter value is never found (e.g. a typo in the status text) you'll get a warning.

In [ ]:
from collections import Counter

wbs = load_workbook(MASTER_FILE)
scope_log = []
for tab, col, op, value in SCOPE_FILTERS:
    if op not in ("equals", "not equals"):
        raise ValueError(f"Filter operator must be 'equals' or 'not equals', not '{op}'")
    ws = wbs[tab]
    values = list(value) if isinstance(value, (list, tuple, set)) else [value]
    targets = {norm_text(v) for v in values}

    hdr, c = find_header(ws, col)
    data_rows = [r for r in range(hdr + 1, ws.max_row + 1) if any(x.value not in (None, "") for x in ws[r])]
    found = Counter(" ".join(str(ws.cell(r, c).value).split()) if ws.cell(r, c).value not in (None, "") else "(blank)"
                    for r in data_rows)
    if not targets & {norm_text(v) for v in found}:
        print(f"WARNING: '{tab}' > '{col}' never contains {values}. Values found: {dict(found)}")

    if op == "equals":
        kept = keep_rows_where(ws, col, lambda v, t=targets: norm_text(v) in t)
    else:
        kept = keep_rows_where(ws, col, lambda v, t=targets: norm_text(v) not in t)
    scope_log.append({"Tab": tab, "Filter": f"{col} {op} {' / '.join(map(str, values))}",
                      "Rows before": len(data_rows), "Rows kept": kept, "Rows removed": len(data_rows) - kept})

meta = wbs["Metadata"]
for e in scope_log:
    meta.append([f"Scope filter – {e['Tab']}", f"{e['Filter']} (kept {e['Rows kept']} of {e['Rows before']} rows)"])

IN_SCOPE_FILE = OUTPUT_DIR / f"Inventory_Attestation_IN_SCOPE_{STAMP}.xlsx"
wbs.save(IN_SCOPE_FILE)
print(f"In-scope copy saved: {IN_SCOPE_FILE}")
display(pd.DataFrame(scope_log))

## 5. Step 2a – Build the owner list and check the data
Review the output of this cell before generating files: unassigned items and orphan IDs are worth fixing first.

In [ ]:
wbm  = load_workbook(IN_SCOPE_FILE)   # owners and their items: in-scope only
wbmf = load_workbook(MASTER_FILE)     # data-quality checks: full inventory

owners = {}        # normalised name -> {"Owner": display name, "Model": {...}, "DQM": {...}, "PMA": {...}}
unassigned = []    # items with an ID but no owner
for group, cfg in GROUPS.items():
    ws = wbm[cfg["inventory_sheet"]]
    hdr, own_col = find_header(ws, cfg["owner_col"])
    _,   id_col  = find_header(ws, id_col_for(cfg["inventory_sheet"], cfg))
    for r in range(hdr + 1, ws.max_row + 1):
        item_id = norm_id(ws.cell(r, id_col).value)
        if not item_id:
            continue
        names = split_owners(ws.cell(r, own_col).value)
        if not names:
            unassigned.append({"Group": group, "ID": ws.cell(r, id_col).value, "Tab": ws.title, "Row": r})
        for n in names:
            entry = owners.setdefault(norm_text(n), {"Owner": n, **{g: set() for g in GROUPS}})
            entry[group].add(item_id)

# IDs in limitation / validation tabs that are not in the (full) inventory tab
orphans = []
for group, cfg in GROUPS.items():
    inv_ws = wbmf[cfg["inventory_sheet"]]
    hdr, id_col = find_header(inv_ws, id_col_for(cfg["inventory_sheet"], cfg))
    inventory_ids = {norm_id(inv_ws.cell(r, id_col).value) for r in range(hdr + 1, inv_ws.max_row + 1)}
    for sheet in cfg["sheets"][1:]:
        ws = wbmf[sheet]
        hdr, id_col = find_header(ws, id_col_for(sheet, cfg))
        for r in range(hdr + 1, ws.max_row + 1):
            v = norm_id(ws.cell(r, id_col).value)
            if v and v not in inventory_ids:
                orphans.append({"Group": group, "Tab": sheet, "Row": r, "ID": ws.cell(r, id_col).value})

summary = pd.DataFrame(
    [{"Owner": o["Owner"], **{f"# {g}s": len(o[g]) for g in GROUPS}} for o in owners.values()]
).sort_values("Owner").reset_index(drop=True)

print(f"{len(owners)} distinct owners with in-scope items")
display(summary)
if unassigned:
    print(f"\n{len(unassigned)} item(s) with no owner (in scope, but no file will include them):")
    display(pd.DataFrame(unassigned))
if orphans:
    print(f"\n{len(orphans)} row(s) whose ID is not in the inventory tab at all, retired items included (data-quality check):")
    display(pd.DataFrame(orphans))

## 6. Step 2b – Generate one file per owner

In [ ]:
OWNER_DIR = OUTPUT_DIR / "Owner files"
OWNER_DIR.mkdir(exist_ok=True)

log, used_names = [], set()
for key, o in sorted(owners.items()):
    wbo = load_workbook(IN_SCOPE_FILE)
    row_counts = {}
    for group, cfg in GROUPS.items():
        for sheet in cfg["sheets"]:
            if o[group]:
                row_counts[sheet] = filter_sheet(wbo[sheet], id_col_for(sheet, cfg), o[group])
            else:
                del wbo[sheet]                      # owner has nothing in this group

    clean_references(wbo)
    add_cover_page(wbo, o["Owner"], o, row_counts)
    meta = wbo["Metadata"]
    for row in meta.iter_rows(min_col=1, max_col=2):
        if row[0].value == "Tabs included":
            row[1].value = ", ".join(s for s in wbo.sheetnames if s not in ("Metadata", "Attestation"))
    meta.append(["Owner", o["Owner"]])
    for g in GROUPS:
        meta.append([f"{g} IDs included", ", ".join(sorted(o[g])).upper() or "None"])
    reset_tab_selection(wbo)

    fname = f"Inventory_Attestation_{safe_filename(o['Owner'])}_{STAMP}"
    n = 2
    while fname.casefold() in used_names:            # avoid two owners overwriting one file
        fname = f"Inventory_Attestation_{safe_filename(o['Owner'])}_{n}_{STAMP}"; n += 1
    used_names.add(fname.casefold())
    path = OWNER_DIR / f"{fname}.xlsx"
    wbo.save(path)

    log.append({"Owner": o["Owner"], **{f"# {g}s": len(o[g]) for g in GROUPS},
                "Tabs": len(wbo.sheetnames) - 2, "File": path.name,
                "Date sent": None, "Response received": None, "Changes requested": None})
    print(f"✓ {o['Owner']:<35} {path.name}")

log_df = pd.DataFrame(log)
LOG_FILE = OUTPUT_DIR / f"Attestation_Distribution_Log_{STAMP}.xlsx"
wb_log = Workbook()                                   # written with openpyxl (works with old pandas)
wb_log.remove(wb_log.active)
for title, records in [("Distribution log", log), ("No owner", unassigned), ("Orphan IDs", orphans)]:
    if not records:
        continue
    ws = wb_log.create_sheet(title)
    headers = list(records[0].keys())
    ws.append(headers)
    for c in ws[1]:
        c.font = Font(bold=True, color="FFFFFF")
        c.fill = PatternFill("solid", fgColor="1F4E78")
    for rec in records:
        ws.append([rec[h] for h in headers])
    for i, h in enumerate(headers, 1):
        ws.column_dimensions[get_column_letter(i)].width = max(14, len(str(h)) + 4)
wb_log.save(LOG_FILE)

print(f"\n{len(log)} owner files saved in: {OWNER_DIR}")
print(f"Distribution log: {LOG_FILE}")